# SPECTER2 Embeddings — Replace TF-IDF

**Why TF-IDF fails**: 5,000 bag-of-words features treat 'cancer' and 'tumor' as unrelated.
They carry no understanding of scientific language or citation relationships.

**SPECTER2** (AllenAI) is a transformer model trained specifically to predict **citation relationships**
between scientific papers. Its embeddings encode:
- Semantic meaning of scientific text
- Which topics cite each other
- Research quality signals implicit in language

This is the exact task we need.

**Input**: title + abstract (concatenated)  
**Output**: 768-dim dense embedding per paper  
**Replaces**: 5,000 sparse TF-IDF features

### Setup
```bash
pip install sentence-transformers
```

**Runtime**: ~5–15 min on CPU, ~1–2 min on GPU (6,000 papers)

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import time

# Install if needed:
# !pip install sentence-transformers
from sentence_transformers import SentenceTransformer

print("sentence-transformers loaded successfully")

## 1. Load Data

In [ ]:
df = pd.read_pickle('../data/processed/cleaned_data.pkl')
print(f"Dataset: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Abstracts available: {df['Abstract'].notna().sum()}")
print(f"Titles available:    {df['Title'].notna().sum() if 'Title' in df.columns else 'N/A'}")

## 2. Prepare Input Text

SPECTER uses title + abstract separated by `[SEP]` token.

In [ ]:
def build_specter_input(row):
    title = str(row.get('Title', '') or '').strip()
    abstract = str(row.get('Abstract', '') or '').strip()
    if title and abstract:
        return title + ' [SEP] ' + abstract
    elif title:
        return title
    else:
        return abstract

# Try common column name variants
title_col = next((c for c in df.columns if c.lower() in ['title', 'paper_title', 'article_title']), None)
abstract_col = next((c for c in df.columns if c.lower() in ['abstract', 'abstract_text']), None)

print(f"Title column   : {title_col}")
print(f"Abstract column: {abstract_col}")

if title_col:
    df['_title_'] = df[title_col].fillna('')
else:
    df['_title_'] = ''

if abstract_col:
    df['_abstract_'] = df[abstract_col].fillna('')
else:
    df['_abstract_'] = ''

df['specter_input'] = df['_title_'] + ' [SEP] ' + df['_abstract_']
df['specter_input'] = df['specter_input'].str.strip()

print(f"\nSample input:")
print(df['specter_input'].iloc[0][:200])
print(f"\nEmpty inputs: {(df['specter_input'].str.strip() == '[SEP]').sum()}")

## 3. Load SPECTER2 Model

In [ ]:
# SPECTER2 — trained on 146M citation pairs from Semantic Scholar
# allenai/specter2_base is the base model (768-dim)
print("Loading SPECTER2... (downloads ~440MB on first run)")
t0 = time.time()

model = SentenceTransformer('allenai/specter2_base')

print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 4. Generate Embeddings

In [ ]:
texts = df['specter_input'].tolist()

print(f"Encoding {len(texts)} papers...")
t0 = time.time()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2 normalize — better for cosine similarity
)

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(texts)*1000:.0f}ms/paper)")
print(f"Embeddings shape: {embeddings.shape}")

## 5. Create Feature DataFrame

In [ ]:
n_dims = embeddings.shape[1]
col_names = [f'specter_{i}' for i in range(n_dims)]

specter_df = pd.DataFrame(embeddings, columns=col_names, index=df.index)

print(f"SPECTER features: {specter_df.shape}")
print(f"Memory usage: {specter_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nFirst 5 columns of first row:")
print(specter_df.iloc[0, :5])

## 6. Quick Sanity Check — Does SPECTER Separate Classes?

Check if high-impact papers cluster separately in embedding space.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Load target — align to papers that have embeddings AND a label
y = pd.read_pickle('../data/features/y_classification.pkl')
y = y.reindex(df.index)

valid_mask = y.notna()
y_valid = y[valid_mask].astype(int)
X_specter_valid = specter_df[valid_mask]

print(f"Papers with labels: {valid_mask.sum()} / {len(y)}")
print(f"High-impact: {y_valid.sum()} ({y_valid.mean()*100:.1f}%)")

# Quick LR on SPECTER alone (5-fold CV)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_specter_valid)

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
cv_f1  = cross_val_score(lr, X_scaled, y_valid, cv=5, scoring='f1').mean()
cv_auc = cross_val_score(lr, X_scaled, y_valid, cv=5, scoring='roc_auc').mean()

print(f"\nSPECTER-only LR (5-fold CV):")
print(f"  F1      : {cv_f1:.4f}")
print(f"  ROC-AUC : {cv_auc:.4f}")
print(f"\nTF-IDF baseline LR was: F1≈0.6254  AUC≈0.81")
print(f"Delta vs TF-IDF: F1 {cv_f1 - 0.6254:+.4f}  AUC {cv_auc - 0.8104:+.4f}")

# PCA visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2196F3' if yi == 0 else '#F44336' for yi in y_valid]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.3, s=5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2196F3', label='Low impact'),
                   Patch(color='#F44336', label='High impact')])
ax.set_title('SPECTER2 embeddings — PCA (colored by citation impact)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

## 7. Save SPECTER Features (replaces text_features.pkl)

In [ ]:
feature_dir = Path('../data/features')
feature_dir.mkdir(parents=True, exist_ok=True)

# Save alongside TF-IDF (don't overwrite yet — run nb23 to merge)
specter_df.to_pickle(feature_dir / 'specter_features.pkl')
print(f"Saved: {feature_dir / 'specter_features.pkl'}")
print(f"Shape: {specter_df.shape}")
print(f"\nNext step: run nb23_specter to combine SPECTER + venue + author features")

## Summary

| | TF-IDF | SPECTER2 |
|---|---|---|
| Features | 5,000 sparse | 768 dense |
| Understands synonyms | ❌ | ✅ |
| Trained on citations | ❌ | ✅ (146M pairs) |
| Cross-paper semantic similarity | ❌ | ✅ |

After saving, **run `23b_feature_engineering_specter.ipynb`** to combine SPECTER + venue + author features and retrain.